# 字幕メーカーの準備（やることは1つだけです）

上のメニュー「ランタイム」→「すべてのセルを実行」を押してください。

- そのあと、このタブは閉じないでください（閉じると処理が止まります）
- Googleドライブの許可の画面が出たら「許可」を押してください
- 「このノートブックは Google が作成したものではありません」という画面が出たら、「そのまま実行」を押してください（配布されたソフトの部品なので大丈夫です）
- 有料プランなどの案内が出ても、押さなくて大丈夫です
- 下のセルが順番に動き、最後に「✅ 準備ができました」と出たら完了です


In [ ]:
# ① GPU 確認
# VERSION = "1.1"
import subprocess
import sys
from IPython.display import Markdown, display

def show_status(text):
    display(Markdown(f"## {text}"))

try:
    subprocess.run(["nvidia-smi"], check=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    show_status("「ランタイム」→「ランタイムのタイプを変更」で「T4 GPU」を選び、すべてのセルを実行してください。")
    raise SystemExit("GPUを使用するバックエンドは利用できません")
show_status("✅ GPU 確認")


In [ ]:
# ② 依存ライブラリと CUDA 12 ランタイム
import ctypes
from importlib.metadata import distribution

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "faster-whisper==1.2.1",
    "nvidia-cublas-cu12", "nvidia-cudnn-cu12",
])
cuda_libraries = []
for package, name in [
    ("nvidia-cublas-cu12", "libcublasLt.so.12"),
    ("nvidia-cublas-cu12", "libcublas.so.12"),
    ("nvidia-cudnn-cu12", "libcudnn.so.9"),
]:
    dist = distribution(package)
    path = next(dist.locate_file(entry) for entry in dist.files if entry.name == name)
    cuda_libraries.append(ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL))
show_status("✅ 依存ライブラリと CUDA 12 ランタイム")


In [ ]:
# ③ Drive マウント
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/字幕メーカー")
for folder in ("jobs", "inbox", "working", "out", "done", "error"):
    (root / folder).mkdir(parents=True, exist_ok=True)
show_status("✅ Drive マウント")


In [ ]:
# ④ 切断された作業の再開
import os

for audio in (root / "working").glob("job-*_???.mp3"):
    if (root / "out" / audio.with_suffix(".srt").name).is_file():
        os.replace(audio, root / "done" / audio.name)
    else:
        os.replace(audio, root / "inbox" / audio.name)
show_status("✅ 切断された作業の再開")


In [ ]:
# ⑤ 既定の GPU モデルをロード
from faster_whisper import WhisperModel

print("初回はモデルのダウンロードに数分かかることがあります。")
active_model = "small"
model = WhisperModel(active_model, device="cuda", compute_type="float16")
show_status("✅ 既定の GPU モデルをロード")


In [ ]:
# ⑥ 30 秒間隔で監視
import json
import re
import tempfile
import time
import wave
from datetime import datetime

import numpy as np

INTERVAL = 30
MAX_ATTEMPTS = 3

def write_text(path, content):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(content, encoding="utf-8")
    os.replace(temporary, path)

def jobs_in_order():
    jobs = []
    for path in (root / "jobs").glob("job-*.json"):
        if not re.fullmatch(r"job-\d{8}-\d{6}", path.stem):
            continue
        try:
            job = json.loads(path.read_text(encoding="utf-8"))
            if job["job_id"] != path.stem:
                raise ValueError("ジョブIDがファイル名と異なります")
            if not 1 <= job["chunk_count"] == len(job["chunks"]) <= 999:
                raise ValueError("チャンク数が不正です")
            if job["model"] not in {"large-v3-turbo", "small"}:
                raise ValueError("未対応のモデルです")
            if job["language"] not in {None, "ja", "en"}:
                raise ValueError("未対応の言語です")
            jobs.append((datetime.fromisoformat(job["issued_at"]), job))
        except (OSError, ValueError, KeyError, TypeError) as exc:
            print(f"{path.name}: ジョブを読み取れません: {exc}")
    return [job for _, job in sorted(jobs, key=lambda item: (item[0], item[1]["job_id"]))]

def stamp(millis):
    hours, millis = divmod(millis, 3600000)
    minutes, millis = divmod(millis, 60000)
    seconds, millis = divmod(millis, 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{millis:03}"

TIMING = re.compile(r"(\d{2,}:\d{2}:\d{2}[,.]\d{3})\s+-->\s+(\d{2,}:\d{2}:\d{2}[,.]\d{3})")

def milliseconds(timestamp):
    hours, minutes, seconds, fraction = map(int, re.split(r"[:,.]", timestamp))
    return ((hours * 60 + minutes) * 60 + seconds) * 1000 + fraction

def parse_srt(content):
    for block in re.split(r"\n[ \t]*\n", content.strip()):
        if not block:
            continue
        lines = block.splitlines()
        if lines[0].strip().isdigit():
            lines = lines[1:]
        match = TIMING.fullmatch(lines[0].strip()) if lines else None
        if not match or len(lines) < 2:
            raise ValueError("SRT の形式が不正です。")
        yield (*map(milliseconds, match.groups()), "\n".join(lines[1:]))

SAMPLE_RATE = 16000

def short_sections(pcm):
    """Return contiguous sample ranges using the v4.3 pause rules."""
    frame_size = SAMPLE_RATE // 50
    pauses = []
    quiet_start = None
    for position in range(0, len(pcm), frame_size):
        frame = pcm[position:position + frame_size].astype(np.int32)
        quiet = np.abs(frame).sum() < 350 * len(frame)
        if quiet:
            if quiet_start is None:
                quiet_start = position
        elif quiet_start is not None:
            if position - quiet_start >= SAMPLE_RATE // 2:
                pauses.append((quiet_start + position) // 2)
            quiet_start = None

    start = index = 0
    while start < len(pcm):
        minimum = start + 3 * SAMPLE_RATE
        limit = start + 24 * SAMPLE_RATE
        while index < len(pauses) and pauses[index] <= minimum:
            index += 1
        end = min(limit, len(pcm))
        if index < len(pauses) and pauses[index] < limit:
            end = pauses[index]
        if len(pcm) - end < 3 * SAMPLE_RATE:
            end = len(pcm)
        yield start, end
        start = end

def load_model(name):
    global model, active_model
    if active_model != name:
        model = None
        active_model = None
        model = WhisperModel(name, device="cuda", compute_type="float16")
        active_model = name

def decode_pcm(audio):
    with tempfile.TemporaryDirectory() as temp:
        wav = Path(temp) / "audio.wav"
        subprocess.run([
            "ffmpeg", "-v", "error", "-i", str(audio),
            "-ac", "1", "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", str(wav),
        ], check=True, capture_output=True)
        with wave.open(str(wav), "rb") as source:
            return np.frombuffer(source.readframes(source.getnframes()), dtype="<i2")

def recognize(pcm, language, start_sample=0):
    samples = pcm.astype(np.float32) / 32768.0
    segments, _ = model.transcribe(
        samples, language=language, beam_size=5, vad_filter=True,
        no_speech_threshold=0.6, condition_on_previous_text=False,
    )
    offset = start_sample / SAMPLE_RATE
    duration = len(pcm) / SAMPLE_RATE
    for segment in segments:
        text = " ".join(segment.text.split())
        start = round((offset + max(0, segment.start)) * 1000)
        end = round((offset + min(duration, segment.end)) * 1000)
        if text and end > start:
            yield start, end, text

def transcribe(audio, job):
    load_model(job["model"])
    pcm = decode_pcm(audio)
    cues = []
    for start, end in short_sections(pcm):
        # SRT stays chunk-relative. The Windows merge adds the chunk offset.
        cues.extend(recognize(pcm[start:end], job["language"], start))
    return "\n".join(
        f"{index}\n{stamp(start)} --> {stamp(end)}\n{text}\n"
        for index, (start, end, text) in enumerate(cues, 1)
    )

def looks_english(text):
    return not re.search(r"[一-龯ぁ-んァ-ン]", text) and len(re.findall(r"[A-Za-z]{3,}", text)) >= 2

def merged_cues_with_sources(job, succeeded):
    """Mirror jobs.merge seam decisions while retaining the source chunk index."""
    cues = []
    for index in succeeded:
        stem = f"{job['job_id']}_{index:03}"
        content = (root / "out" / f"{stem}.srt").read_text(encoding="utf-8-sig")
        chunk = job["chunks"][index - 1]
        offset = round(chunk["start_seconds"] * 1000)
        owned = round(chunk["owned_start_seconds"] * 1000)
        previous_end = max((end for _, end, _, _ in cues), default=0)
        cutoff = max(owned, previous_end)
        for start, end, text in parse_srt(content):
            start, end = start + offset, end + offset
            if end <= cutoff:
                continue
            if start < previous_end and cues and cues[-1][2] == text:
                continue
            cues.append((max(start, cutoff), end, text, index))
    return sorted(cues, key=lambda cue: cue[0])

def repair_english(job, succeeded):
    attempts = 0
    replacements = {}
    if job["language"] is not None:
        return attempts, replacements

    previous_text = ""
    cached_index, pcm = None, None
    for ordinal, (start, end, text, index) in enumerate(merged_cues_with_sources(job, succeeded)):
        if (end - start <= 6500 and looks_english(previous_text)
                and re.search(r"[ァ-ヴー]{5,}", text)
                and not re.search(r"[一-龯ぁ-んA-Za-z]", text)):
            try:
                if cached_index != index:
                    cached_index, pcm = index, None
                    stem = f"{job['job_id']}_{index:03}"
                    pcm = decode_pcm(root / "done" / f"{stem}.mp3")
                if pcm is not None:
                    offset = round(job["chunks"][index - 1]["start_seconds"] * 1000)
                    begin = max(0, (start - offset - 200) * SAMPLE_RATE // 1000)
                    stop = min(len(pcm), (end - offset + 300) * SAMPLE_RATE // 1000)
                    if begin < stop:
                        attempts += 1
                        load_model(job["model"])
                        candidate = list(recognize(pcm[begin:stop], "en"))
                        if len(candidate) == 1 and looks_english(candidate[0][2]):
                            text = replacements[str(ordinal)] = candidate[0][2]
            except Exception:
                print(f"{job['job_id']}: 英語の再確認に失敗したため元の字幕を使用します")
        previous_text = text
    return attempts, replacements

def process_chunk(source, output, error, attempts, job):
    claimed = root / "working" / source.name
    try:
        source.rename(claimed)
    except FileNotFoundError:
        return
    try:
        write_text(output, transcribe(claimed, job))
    except Exception as exc:
        attempts.append(str(exc))
        write_text(error, json.dumps({"attempts": attempts}, ensure_ascii=False))
        if len(attempts) < MAX_ATTEMPTS:
            os.replace(claimed, source)
        else:
            os.replace(claimed, root / "error" / claimed.name)
        print(f"{source.name}: 失敗 {len(attempts)}/{MAX_ATTEMPTS}: {exc}")
    else:
        os.replace(claimed, root / "done" / claimed.name)
        print(f"{source.name}: 完了")

def scan(previous):
    now = time.monotonic()
    inbox = {}
    for audio in (root / "inbox").glob("job-*_???.mp3"):
        try:
            size = audio.stat().st_size
        except FileNotFoundError:
            continue
        old_size, since = previous.get(audio.name, (None, now))
        inbox[audio.name] = (size, since if size == old_size else now)

    for job in jobs_in_order():
        job_id = job["job_id"]
        complete = root / "out" / f"{job_id}_COMPLETE.txt"
        cancelled = root / "jobs" / f"{job_id}_CANCELLED.txt"
        if complete.is_file() or cancelled.is_file():
            continue
        succeeded = []
        failed = []
        for index, chunk in enumerate(job["chunks"], start=1):
            if cancelled.is_file():
                break
            stem = f"{job_id}_{index:03}"
            source = root / "inbox" / f"{stem}.mp3"
            output = root / "out" / f"{stem}.srt"
            error = root / "error" / f"{stem}.json"
            try:
                attempts = json.loads(error.read_text(encoding="utf-8"))["attempts"] if error.is_file() else []
                if not output.is_file() and len(attempts) < MAX_ATTEMPTS and source.name in inbox:
                    size, since = inbox[source.name]
                    if size > 0 and size == chunk.get("size_bytes", size) and now - since >= INTERVAL:
                        if source.stat().st_size == size:
                            inbox.pop(source.name)
                            process_chunk(source, output, error, attempts, job)
                if output.is_file():
                    succeeded.append(index)
                elif len(attempts) >= MAX_ATTEMPTS:
                    failed.append(index)
                    if source.is_file():
                        os.replace(source, root / "error" / source.name)
            except (OSError, ValueError, KeyError) as exc:
                print(f"{stem}: 同期状態を再確認します: {exc}")
        if not cancelled.is_file() and len(succeeded) + len(failed) == job["chunk_count"]:
            attempts, replacements = repair_english(job, succeeded)
            write_text(complete, json.dumps({
                "english_rescue_attempts": attempts,
                "english_rescue_applied": len(replacements),
                "english_replacements": replacements,
                "total": job["chunk_count"], "succeeded": len(succeeded),
                "failed": len(failed), "success_indices": succeeded, "failed_indices": failed,
                "srt_sizes": {
                    str(index): (root / "out" / f"{job_id}_{index:03}.srt").stat().st_size
                    for index in succeeded
                },
            }))
    return inbox

# Keep sending timestamps while the main thread transcribes a chunk.
import threading

def heartbeat():
    write_text(root / "colab_heartbeat.json", json.dumps({"timestamp": time.time()}))

def pulse(stop):
    while not stop.wait(INTERVAL):
        try:
            heartbeat()
        except OSError as exc:
            print(f"Drive の状態を再確認します: {exc}")

heartbeat()
stop = threading.Event()
sender = threading.Thread(target=pulse, args=(stop,), daemon=True)
sender.start()
show_status("✅ 準備ができました")
try:
    previous = {}
    while True:
        try:
            previous = scan(previous)
            print(f"{datetime.now():%H:%M:%S} 待機中 / inbox: {len(previous)} 件")
        except OSError as exc:
            previous = {}
            print(f"Drive の状態を再確認します: {exc}")
        time.sleep(INTERVAL)
finally:
    stop.set()
    sender.join()
    (root / "colab_heartbeat.json").unlink(missing_ok=True)
